# Falcon AI End-to-End Workflow

[Read the step-by-step guide](https://docs.futureagi.com/docs/cookbook/falcon-ai/end-to-end). Trace two support requests, compare revised answers, and save a scored snapshot. Use a fresh Python 3.12 kernel and run the cells in order. The dashboard steps are manual.

## Install

Install tracing in this kernel. Keep the evaluation and dataset SDKs in separate environments below.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fi-instrumentation-otel", "traceai-openai", "openai"], check=True)

In [ ]:
import os
from getpass import getpass

for name in ("FI_API_KEY", "FI_SECRET_KEY", "OPENAI_API_KEY"):
    if not os.environ.get(name):
        value = getpass(f"{name}: ")
        if not value:
            raise ValueError(f"{name} is required")
        os.environ[name] = value
        del value

# For a self-hosted collector, set FI_TRACES_URL before running this cell.
os.environ.setdefault("FI_TRACES_URL", "https://api.futureagi.com")

## Trace a support agent

Run two synthetic Ridgeline Outfitters questions without supplying the reference facts. Keep the printed trace IDs.

In [ ]:
from fi_instrumentation import register, FITracer
from fi_instrumentation.fi_types import ProjectType
from traceai_openai import OpenAIInstrumentor

trace_provider = register(
    verbose=False,
    project_type=ProjectType.OBSERVE,
    project_name="falcon-ai-end-to-end",
)
OpenAIInstrumentor().instrument(tracer_provider=trace_provider)
tracer = FITracer(trace_provider.get_tracer("falcon-ai-end-to-end"))

In [ ]:
import csv
from uuid import UUID
from openai import OpenAI
from opentelemetry.trace import get_current_span

client = OpenAI()
cases = [
    ("Where is order RO-12345?", "RO-12345 has shipped. Tracking is not yet available."),
    ("Can I return opened trail shoes?", "Trail shoes are returnable within 30 days if unworn."),
]
rows = []

@tracer.agent(name="support_reply")
def my_agent(query, prompt):
    reply = client.chat.completions.create(
        model="gpt-4o-mini", messages=[{"role": "system", "content": prompt},
                                      {"role": "user", "content": query}],
    ).choices[0].message.content
    return str(UUID(int=get_current_span().get_span_context().trace_id)), reply

def run_cases(version, instructions, include_source=False):
    for query, source in cases:
        prompt = instructions + ("\nSource: " + source if include_source else "")
        trace_id, answer = my_agent(query, prompt)
        if not answer:
            raise ValueError("The model returned no answer")
        rows.append(dict(version=version, query=query, context=source,
                         agent_output=answer, trace_id=trace_id))
        print(version, trace_id, answer)
    trace_provider.force_flush()
    with open("support-replays.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

run_cases("baseline", "You answer product and order questions for Ridgeline Outfitters.")

## Find the failing traces

In **Tracing**, open `falcon-ai-end-to-end`. Ask Falcon AI to `/analyze-trace-errors` for the two baseline IDs. Check which claims lack a source.

## Lock the failures into a regression dataset

Ask `/build-dataset` to save the two baseline rows with `query`, `agent_output`, `context` and `trace_id`. Use the reference facts from `cases`, not the generated answers.

## Score the baseline

Use `/run-evaluations` with `factual_accuracy` and `completeness`. Map the answer to `output` and the reference fact to `context` for factual accuracy.

## Generate the prompt fix

Open a failing trace and run `/fix-with-falcon`. Review the suggested edit.

## Apply the fix and verify scores recover

Replay the same queries with the reference facts. Compare baseline and revised output columns, not two scores on the same column. Improvement is not guaranteed.

In [ ]:
run_cases("revised", "Answer only from the supplied source. If a detail is missing, say so.",
          include_source=True)

## Save a scored snapshot from Python

Create two isolated Python environments in this notebook’s working directory. Agent Learning Kit scores the answers; the platform SDK uploads the CSV. Both install packages under `fi`.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)

from pathlib import Path

for directory, package in ((".judges", "agent-learning-kit==0.3.7"), (".datasets", "futureagi==0.7.0")):
    subprocess.run([sys.executable, "-m", "virtualenv", "--quiet", directory], check=True)
    python = Path(directory) / "bin" / "python"
    subprocess.run([str(python), "-m", "pip", "install", "-q", package], check=True)

### Score each replay

In [ ]:
%%writefile score_replays.py
import csv
from fi.evals import evaluate

with open("support-replays.csv", newline="") as f:
    rows = list(csv.DictReader(f))
for row in rows:
    for metric in ("factual_accuracy", "completeness"):
        inputs = {"input": row["query"], "output": row["agent_output"]}
        if metric == "factual_accuracy":
            inputs["context"] = row["context"]
        judge = evaluate(metric, model="turing_flash", **inputs)
        if judge.score is None:
            raise RuntimeError(judge.error or judge.reason)
        row[metric] = judge.score
        row[metric + "_id"] = judge.metadata["eval_id"]
with open("scored-replays.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

In [ ]:
subprocess.run([".judges/bin/python", "score_replays.py"], check=True)

### Upload the snapshot

Open **Datasets** and look up the printed ID. Each of the four rows holds the answer, reference, trace ID and recorded scores.

In [ ]:
%%writefile upload_replays.py
from uuid import uuid4
from fi.datasets import Dataset, DatasetConfig
from fi.utils.types import ModelTypes

name = f"falcon-support-replays-{uuid4().hex[:8]}"
dataset = Dataset(dataset_config=DatasetConfig(name=name, model_type=ModelTypes.GENERATIVE_LLM))
dataset.create(source="scored-replays.csv")
print(dataset.dataset_config.id)

In [ ]:
subprocess.run([".datasets/bin/python", "upload_replays.py"], check=True)